# 01 — Rebuilding the dataset from the published supplement

Everything downstream starts here: a single published file, `NIHMS373071-supplement-3.xls`
(Lounkine et al., *Nature* 486:361–367, 2012), plus the raw SEA prediction dump that came
with it. This notebook runs the pipeline and checks the row count at every stage.

The original version of this was thirteen scripts that had to be run in a particular order,
with files renamed by hand in between, absolute paths pointing at one laptop, and one script
that read a `.fasta` which only ever existed on disk as `.fasta.gz`.

In [1]:
import sys, os
from pathlib import Path

# Work from the repository root wherever the notebook is launched from.
_root = Path.cwd().resolve()
while not (_root / "src" / "offtarget").exists():
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root / "src"))

import pandas as pd
from offtarget.pipeline import build, sea_baseline
pd.set_option("display.width", 140)
manifest = build()
pd.Series(manifest.stages, name="rows").to_frame()

excluding 20 ADRA2A rows from pairs.parquet: The PDB-arm sequence for ADRA2A is 'TSSIVHLCAISLDRYWSITQAIEYNLKRTPRR' - a 32-residue fragment of transmembrane helix 3 and the second intracellular loop, pasted where a binding-pocket sequence belonged. It is what copy-pasting from a structure page by hand produces. The rows are kept in data/raw and data/boltz_outputs as the record and excluded here by name.


,rows
safety_targets,73
uniprot_map,117
drugs,656
predictions,4759
confirmed,174
pairs_all,710
pairs_labelled,708
pairs_primary,690
dimer_panel,51
smiles_comparison,220


## What the source file contains

Eight sheets. Four of them matter here.

In [2]:
xl = pd.ExcelFile("data/raw/NIHMS373071-supplement-3.xls")
pd.DataFrame([{"sheet": s, "rows": len(xl.parse(s))} for s in xl.sheet_names])

,sheet,rows
0,S1 Drugs used in study,656
1,S2 Safety targets,73
2,S3 Confirmed predictions,174
3,S4 Comparison to 1NN,322
4,S5 Target-ADR associations,3257
5,S6 Novel off-target and ADRs,116
6,S7 Target Promiscuity,73
7,S8 Drug promiscuity,656


## The premise: how often SEA was wrong

The project exists because SEA's predictions were taken to the bench and most of them
failed. That number is worth deriving rather than quoting, and deriving it turns up a
counting trap.

`Predictions.dat` has 4,759 rows but only 2,768 distinct `Prediction_Id`: one drug–target
prediction appears once per ChEMBL gene entry it maps to. Counting rows inflates every
total by about 1.7×.

In [3]:
pred = pd.read_csv("data/raw/Predictions.dat", sep="\t")
print(f"rows in Predictions.dat        {len(pred):,}")
print(f"distinct Prediction_Id         {pred.Prediction_Id.nunique():,}")

sea = sea_baseline(Path("data/raw/Predictions.dat"), Path("data/raw/pid_confirmation_status.dat"))
assayed = sea[sea.label.notna()]
n_conf = int((assayed.label == "assay_active").sum())
n_dis = int((assayed.label == "assay_inactive").sum())
print(f"\npredictions taken to the bench {len(assayed):,}")
print(f"  confirmed active             {n_conf:,}")
print(f"  not confirmed                {n_dis:,}  ({n_dis / len(assayed):.1%})")

rows in Predictions.dat        4,759
distinct Prediction_Id         2,768

predictions taken to the bench 1,023
  confirmed active             196
  not confirmed                827  (80.8%)


**80.8% of the SEA predictions that were assayed did not confirm.** That is the premise of
the project, and it is a stronger premise than the 46% figure that earlier write-ups of this
work quoted. 46% does not reproduce from these files under any denominator I could construct;
it is not used anywhere in this repository.

## SEA's own scores as a baseline

Before asking whether a structure model can rank these pairs, it is worth asking whether the
thing that generated them can. Within target, SEA's own similarity and E-value barely order
its confirmed predictions above its disproved ones.

In [4]:
from offtarget.metrics import stratified_auc
from sklearn.metrics import roc_auc_score

a = assayed.dropna(subset=["Max Tc", "neg_log_evalue"])
y = (a.label == "assay_active").astype(int)
pd.DataFrame([
    {"score": c,
     "pooled AUC": round(roc_auc_score(y, a[c]), 3),
     "within-target AUC": round(stratified_auc(a, c)[0], 3),
     "targets used": stratified_auc(a, c)[1]}
    for c in ["Max Tc", "neg_log_evalue"]
])

,score,pooled AUC,within-target AUC,targets used
0,Max Tc,0.536,0.536,39
1,neg_log_evalue,0.510,0.580,39


## The label rename

The upstream scripts mapped SEA's bench outcome to `True positive` / `False positive`,
meaning *SEA predicted this and the assay agreed / disagreed*. Those names then travelled
into an analysis that computes ROC curves, where "true positive" means something else
entirely — one name, two meanings, in the same table. Everything is renamed on the way in.

In [5]:
from offtarget.pipeline import LABEL_MAP
pd.Series(LABEL_MAP, name="renamed to").to_frame()

,renamed to
true positive,assay_active
active,assay_active
false positive,assay_inactive
inactive,assay_inactive
inactive inactive,assay_inactive


Case and trailing whitespace are load-bearing in the source workbooks: `all_with_both_AA.xlsx`
carries 576 rows of `'inactive '` and a single `'Inactive'`. A case-sensitive comparison
silently drops one of them.

In [6]:
raw = pd.read_excel("data/boltz_outputs/all_with_both_AA.xlsx")
raw.Label.value_counts().to_frame("rows")

,rows
Label,
inactive,576
True Positive,149
Inactive,1


## The SMILES step, and what re-deriving it cost

The upstream pipeline resolved every drug name through PubChem with
`pcp.get_compounds(name, "name")[0]` and no validation — while all 656 drugs already carry
published SMILES in sheet S1 of the file it was already reading. S1 is authoritative here.

Comparing the two by InChIKey (RDKit, so the comparison is chemical rather than textual):

In [7]:
smi = pd.read_parquet("data/processed/smiles_comparison.parquet")
print(smi.status.value_counts().to_string())
print()
print(smi.name_match.value_counts().to_string())

status
identical             216
different_compound      3
tautomer_or_stereo      1

name_match
exact               205
case_insensitive     14
suffix_stripped       1


In [8]:
smi[smi.status != "identical"][
    ["drug", "matched_S1_entry", "status", "formula_pubchem", "formula_S1"]]

,drug,matched_S1_entry,status,formula_pubchem,formula_S1
51,Anecortave,anecortave,different_compound,C21H28O4,C23H30O5
135,Olmesartan,olmesartan,different_compound,C24H26N6O3,C29H30N6O6
213,Testosterone_Propionate,testosterone,different_compound,C22H32O3,C19H28O2
215,Warfarin,warfarin,tautomer_or_stereo,C19H16O4,C19H16O4


Three of 220 resolved to a genuinely different compound, and one to a different tautomer:

- **Anecortave** — PubChem returned the free alcohol; the drug is anecortave *acetate*.
- **Olmesartan** — PubChem returned the free acid; the drug is the medoxomil prodrug ester.
- **Testosterone_Propionate** — the name had to be truncated to match S1 at all, and then
  matched plain testosterone rather than the propionate ester.
- **Warfarin** — same formula, different tautomer (open-chain keto vs the 4-hydroxycoumarin enol).

Fourteen more matched only because the lookup here is case-insensitive; the original merge
was not, so those fourteen would have gone through as unmatched.

Small numbers, and that is the useful part of the result: the shortcut mostly worked. It is
still a shortcut that reintroduced avoidable error into a table whose correct values were
one sheet away.

## The excluded rows

One target is filtered by name rather than silently dropped.

In [9]:
for note in manifest.notes:
    print("•", note, "\n")

• excluded 20 ADRA2A rows from pairs.parquet: The PDB-arm sequence for ADRA2A is 'TSSIVHLCAISLDRYWSITQAIEYNLKRTPRR' - a 32-residue fragment of transmembrane helix 3 and the second intracellular loop, pasted where a binding-pocket sequence belonged. It is what copy-pasting from a structure page by hand produces. The rows are kept in data/raw and data/boltz_outputs as the record and excluded here by name. 

• dimer Raw sheet: 15 rows had a SMILES string in the label column; recovered by joining to the master table 



## Manifest

Every input carries a content hash, so a rebuilt table can be traced to the exact bytes it
came from.

In [10]:
import json
json.loads(Path("data/processed/manifest.json").read_text())["inputs"]

{'data/raw/NIHMS373071-supplement-3.xls': '5b980542be3ac002a43d942af867cdcfde659cf8a25989e68d67c6b9cacdf247',
 'data/raw/uniprot_reviewed_2025-11-10.fasta.gz': 'c8cd4ab2e5bd3c8df08b1e5283a87f87cebc5927b463affe60842f5f35bcf551',
 'data/raw/Predictions.dat': '23d0b051f35764f071867cc4c67dbdef473b433910dc1665b5571de3f855fc97',
 'data/boltz_outputs/alltargetSHOICHET.xlsx': '740c64916e9e23694a648cf9458cdf03ed344d6a4ad83de3fe82ab1a96ab90b7',
 'data/boltz_outputs/PDBdimerresult.xlsx': 'f052aa49a9414b26b2b63749cb3d69ffb91495e0e6041f499f2e4d2eda2c3d2b',
 'data/boltz_outputs/pubchem_smiles_expanded.xlsx': '29329aceedbc10f6d9b08dd003e82c2628b022478fdf1fcd07b6c6b06b84238c'}